In [5]:
# pip install groq

In [6]:

from groq import Groq
import ollama

import os

import faiss

import numpy as np

from pypdf import PdfReader


In [32]:

client = Groq(api_key="")

print("Groq client is ready")

Groq client is ready


In [33]:
def load_pdfs(data_dir):

    document = ""

    for file_name in os.listdir(data_dir):

        if file_name.endswith(".pdf"):

            file_path = os.path.join(data_dir, file_name)

            reader = PdfReader(file_path)

            for page in reader.pages:

                text = page.extract_text()

                if text:
                    document += text + "\n"

            print("Document loaded:", file_name)

    return document

In [34]:
document = load_pdfs("medical")

print("Total characters:", len(document))

Document loaded: 01_dengue_fever.pdf
Document loaded: 02_diabetes_overview.pdf
Document loaded: 03_hypertension.pdf
Document loaded: 04_asthma.pdf
Document loaded: 05_covid19.pdf
Document loaded: 06_anemia.pdf
Document loaded: 07_tuberculosis.pdf
Document loaded: 08_malaria.pdf
Document loaded: 09_heart_disease.pdf
Document loaded: 10_kidney_disease.pdf
Total characters: 3906


In [35]:
def chunk_document(document):

    chunk_size = 300
    chunk_overlap = 50

    chunks = []

    start = 0

    while start < len(document):

        end = start + chunk_size

        chunk = document[start:end]

        chunks.append(chunk)

        start = end - chunk_overlap

    return chunks

In [36]:
chunks = chunk_document(document)

print("Total chunks:", len(chunks))

Total chunks: 16


In [37]:
print("\nFirst chunk:")
print(chunks[0])


First chunk:
Dengue Fever
Dengue fever is a mosquito-borne viral infection common in tropical and subtropical regions. Symptoms
typically appear 4–10 days after infection and include high fever, severe headache, pain behind the
eyes, muscle and joint pain, nausea, vomiting, swollen glands, and rash.
Treatment: T


In [38]:
def create_embeddings(chunks):

    embeddings = []

    for chunk in chunks:

        response = ollama.embeddings(
            model="nomic-embed-text",
            prompt=chunk
        )

        embeddings.append(response["embedding"])

    return np.array(embeddings, dtype="float32")

In [39]:
chunk_embeddings = create_embeddings(chunks)

print("Embedding shape:", chunk_embeddings.shape)

Embedding shape: (16, 768)


In [40]:
def create_faiss_index(embeddings):

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)

    return index

In [41]:
index = create_faiss_index(chunk_embeddings)

print("Vectors stored in FAISS:", index.ntotal)

Vectors stored in FAISS: 16


In [42]:
def embed_question(question):

    response = ollama.embeddings(
        model="nomic-embed-text",
        prompt=question
    )

    question_embedding = np.array(
        [response["embedding"]],
        dtype="float32"
    )

    return question_embedding

In [43]:
question = "Dengue fever? Prevention and Treatment"

question_embedding = embed_question(question)

print("Question embedding shape:", question_embedding.shape)

Question embedding shape: (1, 768)


In [44]:
def search_faiss(question_embedding, index, k=3):

    distances, indices = index.search(
        question_embedding,
        k
    )

    return distances[0], indices[0]

In [45]:
distances, indices = search_faiss(
    question_embedding,
    index
)

print("Distances:", distances)
print("Chunk numbers:", indices)

Distances: [125.81851 200.64975 274.94012]
Chunk numbers: [1 0 8]


In [46]:
def retrieve_chunks(indices, chunks):

    retrieved_chunks = []

    for i in indices:

        retrieved_chunks.append(chunks[i])

    return retrieved_chunks

In [47]:
retrieved_chunks = retrieve_chunks(
    indices,
    chunks
)

print("Number of retrieved chunks:", len(retrieved_chunks))

Number of retrieved chunks: 3


In [48]:
for i, chunk in enumerate(retrieved_chunks):

    print("\n--- Chunk", i + 1, "---")
    print(chunk)


--- Chunk 1 ---
, vomiting, swollen glands, and rash.
Treatment: There is no specific antiviral treatment for dengue fever. Management focuses on symptom
relief, hydration, and monitoring for complications such as severe dengue.
Prevention: Prevent mosquito bites by using repellents, wearing long sleeves, and remov

--- Chunk 2 ---
Dengue Fever
Dengue fever is a mosquito-borne viral infection common in tropical and subtropical regions. Symptoms
typically appear 4–10 days after infection and include high fever, severe headache, pain behind the
eyes, muscle and joint pain, nausea, vomiting, swollen glands, and rash.
Treatment: T

--- Chunk 3 ---
ention: Vaccination, hand hygiene, wearing masks in high-risk areas, and maintaining ventilation.
Treatment: Supportive care, antiviral medications in some cases, and monitoring oxygen levels.

Anemia
Anemia occurs when the body does not have enough healthy red blood cells to carry oxygen to tissues


In [49]:
def create_context(retrieved_chunks):

    context = "\n\n".join(retrieved_chunks)

    return context

In [50]:
context = create_context(retrieved_chunks)

print("CONTEXT:")
print(context)

CONTEXT:
, vomiting, swollen glands, and rash.
Treatment: There is no specific antiviral treatment for dengue fever. Management focuses on symptom
relief, hydration, and monitoring for complications such as severe dengue.
Prevention: Prevent mosquito bites by using repellents, wearing long sleeves, and remov

Dengue Fever
Dengue fever is a mosquito-borne viral infection common in tropical and subtropical regions. Symptoms
typically appear 4–10 days after infection and include high fever, severe headache, pain behind the
eyes, muscle and joint pain, nausea, vomiting, swollen glands, and rash.
Treatment: T

ention: Vaccination, hand hygiene, wearing masks in high-risk areas, and maintaining ventilation.
Treatment: Supportive care, antiviral medications in some cases, and monitoring oxygen levels.

Anemia
Anemia occurs when the body does not have enough healthy red blood cells to carry oxygen to tissues


In [56]:
def generate_answer(question, context):

    prompt = f"""
You are a helpful agriculture assistant.

Answer the question only using the context below.

If the answer is not available in the context,

say:

I don't know based on the provided documents.

CONTEXT:
{context}

QUESTION:
{question}

Give a concise and accurate answer.
"""

    response = client.chat.completions.create(

        model="openai/gpt-oss-20b",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [57]:
answer = generate_answer(
    question,
    context
)

print("ANSWER:")
print(answer)

ANSWER:
**Prevention:**  
- Avoid mosquito bites by using insect repellent, wearing long sleeves and long‑sleeved clothing, and staying indoors during peak mosquito activity.  
- Eliminate standing water around the home to reduce mosquito breeding sites.  

**Treatment:**  
- There is no specific antiviral therapy for dengue.  
- Management is supportive: provide symptom relief (e.g., pain and fever control), maintain adequate hydration, and monitor for signs of severe dengue or complications.


| Model             | Groq model ID             | 
| ----------------- | ------------------------- | 
| **GPT-OSS 20B**   | `openai/gpt-oss-20b`      | 
| **GPT-OSS 120B**  | `openai/gpt-oss-120b`     | 
| **Llama 3.3 70B** | `llama-3.3-70b-versatile` | 
| **Llama 3.1 8B**  | `llama-3.1-8b-instant`    | 
| **Qwen 3.6 27B**  | `qwen/qwen3.6-27b`        |